In [298]:
import numpy as np
import pandas as pd
import os

### get data

In [305]:
from numpy import genfromtxt
my_data = genfromtxt(os.getcwd()+'//Documents//MQE//1st Semester/Econometrics - Advanced Methods/Tutorial/Session_2/iv_data.csv', delimiter=',')[1:,:]

Z = my_data[:,0]
X1 = my_data[:,1]
X2 = my_data[:,2]
Y = my_data[:, 3]

### use OLS -> biased

In [214]:


from statsmodels.regression.linear_model import OLS

lm = OLS(Y, np.array([np.ones(len(X1)), X1, X2]).T).fit()
lm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.655
Model:                            OLS   Adj. R-squared:                  0.654
Method:                 Least Squares   F-statistic:                     1893.
Date:                Sun, 12 Sep 2021   Prob (F-statistic):               0.00
Time:                        15:30:58   Log-Likelihood:                -2373.4
No. Observations:                2000   AIC:                             4753.
Df Residuals:                    1997   BIC:                             4770.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0129      0.018     -0.729      0.466      -0.048       0.022
x1            -0.4313      0.018    -24.265      0.000      -0.466      -0.396
x2             0.9791      0.017     56.852      0.000       0.945       1.013
==============================================================================
Omnibus:                        1.717   Durbin-Watson:                   2.059
Prob(Omnibus):                  0.424   Jarque-Bera (JB):                1.750
Skew:                           0.048   Prob(JB):                        0.417
Kurtosis:                       2.892   Cond. No.                         1.05
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### use IV in two separate steps using OLS

In [220]:
EXOG = np.array([np.ones(len(Z)), Z, X2]).T

iv1st = OLS(X1, EXOG).fit()
iv1st.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     9.636
Date:                Sun, 12 Sep 2021   Prob (F-statistic):           6.84e-05
Time:                        15:35:22   Log-Likelihood:                -2824.7
No. Observations:                2000   AIC:                             5655.
Df Residuals:                    1997   BIC:                             5672.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0381      0.022     -1.715      0.087      -0.082       0.005
x1             0.0949      0.022      4.351      0.000       0.052       0.138
x2             0.0130      0.022      0.602      0.547      -0.029       0.055
==============================================================================
Omnibus:                        0.000   Durbin-Watson:                   1.923
Prob(Omnibus):                  1.000   Jarque-Bera (JB):                0.006
Skew:                          -0.001   Prob(JB):                        0.997
Kurtosis:                       2.992   Cond. No.                         1.03
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [223]:
X_hat = np.hstack((np.ones((len(Y),1)), X2.reshape(-1,1), iv1st.fittedvalues.reshape(-1,1)))
stage_2 = OLS(Y, X_hat).fit()
stage_2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.562
Model:                            OLS   Adj. R-squared:                  0.561
Method:                 Least Squares   F-statistic:                     1279.
Date:                Sun, 12 Sep 2021   Prob (F-statistic):               0.00
Time:                        15:35:59   Log-Likelihood:                -2611.9
No. Observations:                2000   AIC:                             5230.
Df Residuals:                    1997   BIC:                             5247.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0448      0.021     -2.099      0.036      -0.087      -0.003
x1             0.9901      0.020     50.579      0.000       0.952       1.029
x2            -1.3062      0.207     -6.321      0.000      -1.711      -0.901
==============================================================================
Omnibus:                        5.387   Durbin-Watson:                   2.079
Prob(Omnibus):                  0.068   Jarque-Bera (JB):                5.430
Skew:                           0.117   Prob(JB):                       0.0662
Kurtosis:                       2.896   Cond. No.                         10.7
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [264]:
beta = stage_2.params.reshape(-1,1)
X = np.hstack((np.ones((len(Y), 1)), X2.reshape(-1,1), X1.reshape(-1,1)))
y_hat = X@beta

#calculate SE

u_hat = Y - y_hat.ravel()  #.ravel() flatten
k = np.shape(X)[1] # 3 columns
s2 = (u_hat.T @ u_hat)/(len(Y))
var = s2 * np.linalg.inv(X_hat.T @ X_hat)
se = np.sqrt(np.diag(var))
print(se)

# T-stat
t_stats = beta / se.reshape(-1, 1)
print(t_stats)

# p-values
from scipy.stats import t
df = len(Y) - k - 1 # N minus number of parameters
p_values = 2 * (1-t.cdf(abs(t_stats),df))
print(p_values)

# calculate R-squared
tss = (Y-np.mean(Y)).T @ (Y-np.mean(Y))
rss = u_hat.T @ u_hat
r_sq = 1 - rss / tss
print(tss, rss, r_sq)

# create dataframe for betas
pindex = np.arange(1, len(beta)+1)
df = pd.DataFrame(index=pindex)
df['beta'] = beta
df['se'] = se
df['t_stats'] = t_stats
df['p_values'] = p_values

df

[0.02818487 0.02583044 0.2726612 ]
[[-1.59095455]
 [38.33139843]
 [-4.79062647]]
[[1.11778160e-01]
 [0.00000000e+00]
 [1.78511665e-06]]
3639.5567289967203 2782.144123208319 0.23558160227516378


,beta,se,t_stats,p_values
1,-0.044841,0.028185,-1.590955,0.111778
2,0.990117,0.025830,38.331398,0.000000
3,-1.306218,0.272661,-4.790626,0.000002


### 2SLS using linearmodels

In [292]:
from linearmodels.iv import IV2SLS as LM_IV2SLS
#https://bashtage.github.io/linearmodels/iv/iv/linearmodels.iv.model.IV2SLS.html

# formula method
col_names = ['Z', "X1", "X2", "Y"]
data = pd.DataFrame(my_data, columns=col_names)

formula = 'Y ~ 1 + X2 + [X1 ~ Z]'
mod = LM_IV2SLS.from_formula(formula, data).fit(cov_type='unadjusted')
mod.summary

<class 'linearmodels.compat.statsmodels.Summary'>
"""
                          IV-2SLS Estimation Summary                          
==============================================================================
Dep. Variable:                      Y   R-squared:                      0.2356
Estimator:                    IV-2SLS   Adj. R-squared:                 0.2348
No. Observations:                2000   F-statistic:                    1469.4
Date:                Sun, Sep 12 2021   P-value (F-stat)                0.0000
Time:                        16:14:56   Distribution:                  chi2(2)
Cov. Estimator:            unadjusted                                         
                                                                              
                             Parameter Estimates                              
==============================================================================
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept     -0.0448     0.0282    -1.5910     0.1116     -0.1001      0.0104
X2             0.9901     0.0258     38.331     0.0000      0.9395      1.0407
X1            -1.3062     0.2727    -4.7906     0.0000     -1.8406     -0.7718
==============================================================================

Endogenous: X1
Instruments: Z
Unadjusted Covariance (Homoskedastic)
Debiased: False
"""

In [293]:
#non-formula method

mod = LM_IV2SLS(dependent = Y, exog=np.array([np.ones(len(Y)),X2]).T, endog=X1, instruments=Z).fit(cov_type='unadjusted')
mod.summary

<class 'linearmodels.compat.statsmodels.Summary'>
"""
                          IV-2SLS Estimation Summary                          
==============================================================================
Dep. Variable:              dependent   R-squared:                      0.2356
Estimator:                    IV-2SLS   Adj. R-squared:                 0.2348
No. Observations:                2000   F-statistic:                    1469.4
Date:                Sun, Sep 12 2021   P-value (F-stat)                0.0000
Time:                        16:15:01   Distribution:                  chi2(2)
Cov. Estimator:            unadjusted                                         
                                                                              
                             Parameter Estimates                              
==============================================================================
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
exog.0        -0.0448     0.0282    -1.5910     0.1116     -0.1001      0.0104
exog.1         0.9901     0.0258     38.331     0.0000      0.9395      1.0407
endog         -1.3062     0.2727    -4.7906     0.0000     -1.8406     -0.7718
==============================================================================

Endogenous: endog
Instruments: instruments
Unadjusted Covariance (Homoskedastic)
Debiased: False
"""

### 2SLS using statsmodels

In [294]:
from statsmodels.sandbox.regression.gmm import IV2SLS as SM_IV2SLS
#https://www.statsmodels.org/dev/generated/statsmodels.sandbox.regression.gmm.IV2SLS.html

sm_iv2sls = SM_IV2SLS(Y, np.array([np.ones(len(Y)),X1, X2]).T, np.array([np.ones(len(Y)),X2, Z]).T).fit()
sm_iv2sls.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                          IV2SLS Regression Results                           
==============================================================================
Dep. Variable:                      y   R-squared:                       0.236
Model:                         IV2SLS   Adj. R-squared:                  0.235
Method:                     Two Stage   F-statistic:                     733.6
                        Least Squares   Prob (F-statistic):          1.37e-239
Date:                Sun, 12 Sep 2021                                         
Time:                        16:15:11                                         
No. Observations:                2000                                         
Df Residuals:                    1997                                         
Df Model:                           2                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0448      0.028     -1.590      0.112      -0.100       0.010
x1            -1.3062      0.273     -4.787      0.000      -1.841      -0.771
x2             0.9901      0.026     38.303      0.000       0.939       1.041
==============================================================================
Omnibus:                        0.886   Durbin-Watson:                   1.926
Prob(Omnibus):                  0.642   Jarque-Bera (JB):                0.948
Skew:                          -0.042   Prob(JB):                        0.622
Kurtosis:                       2.934   Cond. No.                         1.05
==============================================================================
"""